# 01 — Dataset Analysis, Denoising Impact & Inter-Annotator Agreement

Addresses reviewer points **#4 (dataset distribution analysis)** and the
**inter-annotator agreement** concern raised under *Weak aspects*.

Also produces `silver_standard_bio_naive.csv`, the input required by the
**−HD (no heuristic denoising)** ablation in notebook 02.

Nothing here retrains a model. Run this first — it is cheap and several outputs
feed the later notebooks.

In [ ]:
# --- one identical bootstrap cell in every notebook -----------------------
# Colab gives each notebook its own VM, so nothing is shared in memory.
# The notebooks hand off through files on Drive; config.py holds the paths.
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pytorch-crf seqeval safetensors
!cp /content/drive/MyDrive/crypto-ner/code/*.py /content/

import sys; sys.path.append('/content')
from config import *
bootstrap()

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ROOT = /content/drive/MyDrive/crypto-ner
device: CPU only

  Inputs:
    [ok     ] gold_standard_bio_labeled.csv         0.9 MB  nb 01, 02, 03
    [ok     ] silver_standard_bio.csv              17.5 MB  nb 01
    [ok     ] silver_standard_dataset.csv          14.3 MB  nb 01
    [ok     ] lkb_cleaned.json                      0.2 MB  nb 01, 02, 03
    [ok     ] lkb_case_sensitive.json               0.0 MB  nb 01, 02, 03
    [ok     ] dataset_mentah_twitter.csv           16.4 MB  nb 01, funnel only

  01 → 02
    [ok     ] silver_standard_bio_naive.csv      naive silver labels, required by the -HD ablation

  01 → 03
    [ok     ] gold_strata.csv                    per-sentence stratum, used in the error breakdown

  02 → 03
    [ok     ] oof_proposed.jsonl                 out-of-fold predictions, proposed system
    [ok     ] oof_naive_silver

True

In [ ]:
import os, json, re
import numpy as np
import pandas as pd
from collections import Counter

import utils
from utils import (load_bio_csv, load_lkb, corpus_statistics, assign_strata,
                   extract_entities, gazetteer_label_tokens, TOKEN_PATTERN,
                   fleiss_kappa, token_kappa_table, pairwise_entity_agreement,
                   FINANCIAL_KEYWORDS, BLUE_CHIPS)

lkb_unamb = load_lkb(P_LKB_UNAMB)
lkb_amb   = load_lkb(P_LKB_AMB)
print(f'LKB unambiguous surface forms : {len(lkb_unamb)}')
print(f'LKB ambiguous surface forms   : {len(lkb_amb)}')

LKB unambiguous surface forms : 1672
LKB ambiguous surface forms   : 286


## A. Preprocessing funnel

Reviewers like seeing where the 60,283 → 27,226 attrition happened. This
re-applies each filter **in the original order** but records the count removed at
every stage. Paste the resulting table into Section IV.A.

In [ ]:
blacklist = [
    r'\blistrik\b', r'\bpln\b', r'\bpulsa\b', r'\bmeteran\b', r'\bshopee\b', r'\btokopedia\b',
    r'\btokped\b', r'\bline\b', r'\btiktok\b', r'\broblox\b', r'\blazada\b', r'\bgopay\b',
    r'\bmlbb\b', r'mobile legend', r'\bwebtoon\b', r'\bhayday\b', r'\bovo\b', r'\bsupercell\b',
    r'\bmakeup\b', r'\bseventeen\b', r'\bnana\b', r'\bpreloved\b', r'\bbarenbliss\b',
    r'\bparfum\b', r'\bwts\b', r'\bwtb\b', r'flash\s?sale', r'cashback',
    r'\bgiveaway\b', r'rt\s?(&|dan)?\s?follow', r'link (in|di) bio', r'\bjoin grup\b'
]
blacklist_join = '|'.join(blacklist)

indo_func_words = ['dari','yang','yg','dan','ini','itu','ada','udah','saja','aja',
                   'buat','kalo','kalau','sama','kok','sih','dong']
idfw_join = r'\b(' + '|'.join(indo_func_words) + r')\b'

def standardize(text):
    text = str(text)
    text = re.sub(r'@\w+', '@USER', text)
    text = re.sub(r'http\S+|www\.\S+', 'HTTPURL', text)
    text = re.sub(r'[^\w\s\$\.,!?\-@#]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

df = pd.read_csv(P_RAW)
funnel = [('Raw scraped corpus', len(df))]

df = df.dropna(subset=['full_text'])
df = df[df['full_text'].str.strip() != '']
funnel.append(('After empty / NaN removal', len(df)))

df['clean_text'] = df['full_text'].apply(standardize)

mask = df['clean_text'].str.contains(blacklist_join, case=False, regex=True, na=False)
df = df[~mask]
funnel.append(('After global noise filtering', len(df)))

keep = (df['lang'] == 'id') | df['clean_text'].str.contains(idfw_join, case=False, regex=True, na=False)
df = df[keep]
funnel.append(('After hybrid language filtering', len(df)))

df = df[df['clean_text'].str.len() >= 15]
df = df[df['clean_text'].str.count('@USER') < 5]
df = df[df['clean_text'].str.count('#') < 6]
funnel.append(('After spam filtering', len(df)))

rows = []
prev = funnel[0][1]
for stage, n in funnel:
    rows.append({'Stage': stage, 'Retained': n,
                 'Removed': prev - n,
                 '% of raw': round(100 * n / funnel[0][1], 2)})
    prev = n
df_funnel = pd.DataFrame(rows)
df_funnel.to_csv(f'{OUT}/table_funnel.csv', index=False)
display(df_funnel)

/tmp/ipykernel_3029/361613044.py:31: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df['clean_text'].str.contains(blacklist_join, case=False, regex=True, na=False)
/tmp/ipykernel_3029/361613044.py:35: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  keep = (df['lang'] == 'id') | df['clean_text'].str.contains(idfw_join, case=False, regex=True, na=False)


,Stage,Retained,Removed,% of raw
0,Raw scraped corpus,46207,0,100.00
1,After empty / NaN removal,46207,0,100.00
2,After global noise filtering,37796,8411,81.80
3,After hybrid language filtering,27835,9961,60.24
4,After spam filtering,27226,609,58.92


## B. Gold Standard statistics

Everything the reviewer means by "dataset distribution": size, tag balance,
entity density, span lengths, and the long-tail evidence that supports the OOV
claim made in the Introduction.

In [ ]:
gold_ids, gold_sents, gold_labels = load_bio_csv(P_GOLD_BIO)
stats = corpus_statistics(gold_sents, gold_labels)

for k, v in stats.items():
    if k not in ('top_surface_forms',):
        print(f'{k:32s}: {v}')

print('\nMost frequent entity surface forms:')
for form, n in stats['top_surface_forms']:
    print(f'  {form:24s} {n}')

print(f"\nLong tail: {stats['hapax_ratio']:.1%} of distinct surface forms occur exactly once.")

sentences                       : 1461
tokens                          : 50242
avg_tokens_per_sentence         : 34.388774811772755
entities                        : 1182
entities_per_sentence           : 0.8090349075975359
sentences_without_entity        : 847
unique_surface_forms            : 298
hapax_surface_forms             : 160
hapax_ratio                     : 0.5369127516778524
tag_distribution                : {'O': 49045, 'B-CRYPTO': 1182, 'I-CRYPTO': 15}
span_length_distribution        : {1: 1167, 2: 15}

Most frequent entity surface forms:
  btc                      169
  bitcoin                  165
  eth                      57
  $btc                     50
  pi                       35
  usdt                     32
  sol                      30
  xrp                      29
  ethereum                 27
  $eth                     20
  #bitcoin                 18
  solana                   13
  bnb                      12
  $sol                     7
  #btc             

In [ ]:
# Per-stratum breakdown — reused in notebook 03 for the error analysis
strata = assign_strata(gold_sents, lkb_unamb, lkb_amb)
pd.Series(strata).value_counts().to_frame('sentences').pipe(display)

rows = []
for s in ['Blue-Chip', 'Altcoin', 'Background', 'Other']:
    idx = [i for i, t in enumerate(strata) if t == s]
    if not idx:
        continue
    sub_sents  = [gold_sents[i] for i in idx]
    sub_labels = [gold_labels[i] for i in idx]
    st = corpus_statistics(sub_sents, sub_labels)
    rows.append({'Stratum': s, 'Sentences': st['sentences'], 'Tokens': st['tokens'],
                 'Entities': st['entities'],
                 'Ent./sent.': round(st['entities_per_sentence'], 3),
                 'No-entity sent.': st['sentences_without_entity']})
df_strata = pd.DataFrame(rows)
# df_strata.to_csv(f'{OUT}/table_strata.csv', index=False)
display(df_strata)

# Persist strata so notebook 03 does not have to recompute them
pd.DataFrame({'sentence_id': gold_ids, 'stratum': strata}).to_csv(
    P_GOLD_STRATA, index=False)   # handoff -> notebook 03

,sentences
Altcoin,754
Blue-Chip,387
Background,313
Other,7


,Stratum,Sentences,Tokens,Entities,Ent./sent.,No-entity sent.
0,Blue-Chip,387,13916,623,1.610,64
1,Altcoin,754,26751,438,0.581,528
2,Background,313,9443,113,0.361,255
3,Other,7,132,8,1.143,0


In [ ]:
# Span-length and tag distribution — one compact table for the paper
tag = stats['tag_distribution']
total_tok = sum(tag.values())
df_tags = pd.DataFrame([{'Tag': k, 'Count': v, '%': round(100*v/total_tok, 2)}
                        for k, v in sorted(tag.items())])
display(df_tags)

df_spans = pd.DataFrame([{'Span length (tokens)': k, 'Count': v}
                         for k, v in stats['span_length_distribution'].items()])
display(df_spans)
df_tags.to_csv(f'{OUT}/table_tags.csv', index=False)
df_spans.to_csv(f'{OUT}/table_spans.csv', index=False)

,Tag,Count,%
0,B-CRYPTO,1182,2.35
1,I-CRYPTO,15,0.03
2,O,49045,97.62


,Span length (tokens),Count
0,1,1167
1,2,15


## C. Silver vs Gold

Entity density in the two subsets explains the base model's 42.76% precision far
better than prose can. If silver is markedly denser than gold, distant
supervision is over-labelling — which is exactly the over-prediction behaviour
reported in Section IV.B.

In [ ]:
silver_ids, silver_sents, silver_labels = load_bio_csv(P_SILVER_BIO)
s_stats = corpus_statistics(silver_sents, silver_labels)

comp = pd.DataFrame([
    {'Subset': 'Silver (distant supervision)', **{k: s_stats[k] for k in
        ['sentences', 'tokens', 'entities', 'entities_per_sentence',
         'sentences_without_entity', 'unique_surface_forms']}},
    {'Subset': 'Gold (manual)', **{k: stats[k] for k in
        ['sentences', 'tokens', 'entities', 'entities_per_sentence',
         'sentences_without_entity', 'unique_surface_forms']}},
])
comp['entities_per_sentence'] = comp['entities_per_sentence'].round(3)
comp.to_csv(f'{OUT}/table_silver_vs_gold.csv', index=False)
display(comp)

,Subset,sentences,tokens,entities,entities_per_sentence,sentences_without_entity,unique_surface_forms
0,Silver (distant supervision),25726,850987,37765,1.468,7536,1123
1,Gold (manual),1461,50242,1182,0.809,847,298


## D. Denoising impact + building the −HD silver set

Two outputs:

1. **A number for the paper.** How many candidate spans the heuristic suppressed.
   This quantifies the denoising contribution *directly*, independently of any
   downstream model.
2. **`silver_standard_bio_naive.csv`**, the training input for the −HD ablation.

In [ ]:
df_silver_raw = pd.read_csv(P_SILVER_RAW)
df_silver_raw = df_silver_raw.dropna(subset=['clean_text'])

rows_naive, n_den, n_naive = [], 0, 0

for idx, row in df_silver_raw.iterrows():
    toks = re.findall(TOKEN_PATTERN, str(row['clean_text']))
    if not toks:
        continue
    lab_den   = gazetteer_label_tokens(toks, lkb_unamb, lkb_amb, denoise=True)
    lab_naive = gazetteer_label_tokens(toks, lkb_unamb, lkb_amb, denoise=False)
    n_den   += len(extract_entities(lab_den))
    n_naive += len(extract_entities(lab_naive))
    for t, l in zip(toks, lab_naive):
        rows_naive.append({'Sentence_ID': f'Silver_{idx+1}', 'Token': t, 'Label': l})

pd.DataFrame(rows_naive).to_csv(P_SILVER_BIO_NAIVE, index=False)

suppressed = n_naive - n_den
print(f'Naive longest-match entities     : {n_naive}')
print(f'Heuristic-denoised entities      : {n_den}')
print(f'Suppressed by the heuristic      : {suppressed} ({100*suppressed/max(1,n_naive):.2f}%)')
print(f'\nSaved -> {P_SILVER_BIO_NAIVE}  (handoff to notebook 02)')

Naive longest-match entities     : 53169
Heuristic-denoised entities      : 37765
Suppressed by the heuristic      : 15404 (28.97%)

Saved -> /content/drive/MyDrive/crypto-ner/data/silver_standard_bio_naive.csv  (handoff to notebook 02)


In [ ]:
# Which ambiguous surface forms did the heuristic suppress most often?
sup = Counter()
for idx, row in df_silver_raw.iterrows():
    toks = re.findall(TOKEN_PATTERN, str(row['clean_text']))
    if not toks:
        continue
    d = gazetteer_label_tokens(toks, lkb_unamb, lkb_amb, denoise=True)
    n = gazetteer_label_tokens(toks, lkb_unamb, lkb_amb, denoise=False)
    for t, ld, ln in zip(toks, d, n):
        if ln != 'O' and ld == 'O':
            sup[t.lower()] += 1

df_sup = pd.DataFrame(sup.most_common(20), columns=['Surface form', 'Suppressed count'])
df_sup.to_csv(f'{OUT}/table_denoising_suppressed.csv', index=False)
display(df_sup)

,Surface form,Suppressed count
0,ini,9115
1,amp,2033
2,dia,1375
3,bnb,710
4,meme,443
5,gt,391
6,memecoin,313
7,gas,239
8,max,112
9,cash,110


### Consistency check — run this before trusting the −HD ablation

The cell above re-derived the **denoised** labels as well. They should reproduce
`silver_standard_bio.csv` exactly. If they do not, `utils.FINANCIAL_KEYWORDS` or
`utils.TOKEN_PATTERN` differs from the values used in the original pipeline, and
the −HD ablation would then compare two pipelines differing in more than one
variable rather than isolating the denoising step.

If the check fails, copy the exact `financial_keywords` list and tokenizer regex
out of the original notebook into `utils.py` and re-run section D.

In [ ]:
from utils import extract_entities as _ee

orig_ids, orig_sents, orig_labels = load_bio_csv(P_SILVER_BIO)
orig_n_ent = sum(len(_ee(l)) for l in orig_labels)

print(f'Original silver_standard_bio.csv : {len(orig_sents)} sentences, {orig_n_ent} entities')
print(f'Regenerated (denoise=True)       : {len(df_silver_raw)} sentences, {n_den} entities')

diff = abs(orig_n_ent - n_den)
tol = max(1, int(0.01 * orig_n_ent))
if diff <= tol:
    print(f'\nOK — entity counts agree within {tol} ({diff} apart).')
else:
    print(f'\nMISMATCH — {diff} entities apart (tolerance {tol}).')
    print('Align these with the original notebook before running notebook 02:')
    print(f'  utils.TOKEN_PATTERN       = {utils.TOKEN_PATTERN}')
    print(f'  utils.FINANCIAL_KEYWORDS  = {utils.FINANCIAL_KEYWORDS}')

# Token-level spot check on the first sentences that both files share
n_check = min(200, len(orig_sents))
mismatch = 0
for i in range(n_check):
    toks = orig_sents[i]
    regen = gazetteer_label_tokens(toks, lkb_unamb, lkb_amb, denoise=True)
    if regen != orig_labels[i]:
        mismatch += 1
        if mismatch <= 3:
            print(f'\n  sentence {i}: {" ".join(toks)[:90]}')
            print(f'    original : {orig_labels[i]}')
            print(f'    regen    : {regen}')
print(f'\nToken-level: {n_check - mismatch}/{n_check} sentences match exactly.')

Original silver_standard_bio.csv : 25726 sentences, 37765 entities
Regenerated (denoise=True)       : 25726 sentences, 37765 entities

OK — entity counts agree within 377 (0 apart).

Token-level: 200/200 sentences match exactly.



### D2. Tier shadowing audit


#### A1a
 Read-only. Membuat lkb_unamb_fixed; lkb_unamb dan lkb_amb tidak disentuh.

In [ ]:

import json

with open(P_LKB_AMB, encoding='utf-8') as f:
    _amb_raw = json.load(f)

removal, from_key, from_sym = set(), set(), set()
for k, info in _amb_raw.items():
    k = k.lower().strip()
    if k and ' ' not in k:
        removal.add(k); from_key.add(k)
    if isinstance(info, dict):
        sym = (info.get('symbol') or '').lower().strip()
        if sym and ' ' not in sym:
            removal.add(sym)
            if sym not in from_key:
                from_sym.add(sym)

lkb_unamb_fixed = lkb_unamb - removal
shadowed        = removal & lkb_unamb

print(f'kunci+simbol Ambiguous satu token   : {len(removal)}')
print(f'yang benar-benar ada di Unambiguous : {len(shadowed)}   <- ini yang keluar')
print(f'Unambiguous surface  {len(lkb_unamb)} -> {len(lkb_unamb_fixed)}')
print(f'Ambiguous surface    {len(lkb_amb)} (tidak berubah)')

must_go   = ['ada', 'sol', 'mana', 'pump', 'link']
must_stay = ['near protocol', 'magic eden', 'official trump', 'maple finance',
             'bnb attestation service', 'goatseus maximus']

print('\nguard rail')
bad = []
for t in must_go:
    ok = t not in lkb_unamb_fixed
    print(f'  keluar  {t:26s} {ok}')
    if not ok: bad.append(t)
for t in must_stay:
    if t in lkb_unamb:
        ok = t in lkb_unamb_fixed
        print(f'  tetap   {t:26s} {ok}')
        if not ok: bad.append(t)
    else:
        print(f'  tetap   {t:26s} (memang tidak ada di Unambiguous)')
assert not bad, f'himpunan pembuangan salah: {bad}'
print('  OK — frasa multi-token selamat, ticker satu token tidak')

print(f'\nkeluar HANYA karena simbol (periksa manual, {len(from_sym & lkb_unamb)} item):')
print(sorted(from_sym & lkb_unamb)[:40])

kunci+simbol Ambiguous satu token   : 179
yang benar-benar ada di Unambiguous : 122   <- ini yang keluar
Unambiguous surface  1672 -> 1550
Ambiguous surface    286 (tidak berubah)

guard rail
  keluar  ada                        True
  keluar  sol                        True
  keluar  mana                       True
  keluar  pump                       True
  keluar  link                       True
  tetap   near protocol              True
  tetap   magic eden                 True
  tetap   official trump             True
  tetap   maple finance              True
  tetap   bnb attestation service    True
  tetap   goatseus maximus           True
  OK — frasa multi-token selamat, ticker satu token tidak

keluar HANYA karena simbol (periksa manual, 8 item):
['auction', 'esp', 'jst', 'mmt', 'shfl', 'sn44', 'snt', 'tst']


In [ ]:
# ==========================================================================
# A1b — berapa banyak penekanan tambahan yang dibeli oleh perbaikan tier?
# ==========================================================================
import re, time
import pandas as pd
from collections import Counter
from utils import gazetteer_label_tokens, extract_entities, TOKEN_PATTERN

if 'df_silver_raw' not in globals():
    df_silver_raw = pd.read_csv(P_SILVER_RAW).dropna(subset=['clean_text'])

n_den_old = n_den_fix = n_nai_old = n_nai_fix = 0
naive_sent_diff = 0
new_sup = Counter()

t0 = time.time()
for _, row in df_silver_raw.iterrows():
    toks = re.findall(TOKEN_PATTERN, str(row['clean_text']))
    if not toks:
        continue
    d_old = gazetteer_label_tokens(toks, lkb_unamb,       lkb_amb, denoise=True)
    d_fix = gazetteer_label_tokens(toks, lkb_unamb_fixed, lkb_amb, denoise=True)
    a_old = gazetteer_label_tokens(toks, lkb_unamb,       lkb_amb, denoise=False)
    a_fix = gazetteer_label_tokens(toks, lkb_unamb_fixed, lkb_amb, denoise=False)

    n_den_old += len(extract_entities(d_old))
    n_den_fix += len(extract_entities(d_fix))
    n_nai_old += len(extract_entities(a_old))
    n_nai_fix += len(extract_entities(a_fix))
    if a_old != a_fix:
        naive_sent_diff += 1
    for t, x, y in zip(toks, d_old, d_fix):
        if x != 'O' and y == 'O':
            new_sup[t.lower()] += 1

print(f'{len(df_silver_raw)} baris dalam {time.time()-t0:.0f}s\n')
print(f'entitas naif      tier lama  : {n_nai_old}')
print(f'entitas naif      tier baru  : {n_nai_fix}    kalimat berbeda: {naive_sent_diff}')
print(f'entitas denoised  tier lama  : {n_den_old}    (paper melaporkan 37765)')
print(f'entitas denoised  tier baru  : {n_den_fix}')
print()
print(f'>>> PENEKANAN TAMBAHAN DARI PERBAIKAN : {n_den_old - n_den_fix}')
print(f'total ditekan  lama : {n_nai_old-n_den_old} ({100*(n_nai_old-n_den_old)/n_nai_old:.2f}%)')
print(f'total ditekan  baru : {n_nai_fix-n_den_fix} ({100*(n_nai_fix-n_den_fix)/n_nai_fix:.2f}%)')

assert naive_sent_diff == 0, \
    'SILVER NAIF BERUBAH — naive_silver_ft dan ablation -HD tidak lagi valid, hentikan'
print('\nsilver naif bit-identical -> BASE_NAIVE, naive_silver_ft, ablation -HD tetap valid')

display(pd.DataFrame(new_sup.most_common(25),
                     columns=['Surface form', 'Ditekan setelah perbaikan']))

25726 baris dalam 15s

entitas naif      tier lama  : 53169
entitas naif      tier baru  : 53169    kalimat berbeda: 0
entitas denoised  tier lama  : 37765    (paper melaporkan 37765)
entitas denoised  tier baru  : 22892

>>> PENEKANAN TAMBAHAN DARI PERBAIKAN : 14873
total ditekan  lama : 15404 (28.97%)
total ditekan  baru : 30277 (56.94%)

silver naif bit-identical -> BASE_NAIVE, naive_silver_ft, ablation -HD tetap valid


,Surface form,Ditekan setelah perbaikan
0,ada,5309
1,hype,2055
2,sol,801
3,mana,725
4,gua,694
5,in,478
6,ath,391
7,trump,376
8,on,351
9,up,319


In [ ]:
# ==========================================================================
# A1c — apa yang hilang di sisi gold?
# ==========================================================================
from utils import gazetteer_predict, entity_f1, load_bio_csv

if 'gold_sents' not in globals():
    gold_ids, gold_sents, gold_labels = load_bio_csv(P_GOLD_BIO)

m_old = entity_f1(gold_labels, gazetteer_predict(gold_sents, lkb_unamb,       lkb_amb, denoise=True))
m_fix = entity_f1(gold_labels, gazetteer_predict(gold_sents, lkb_unamb_fixed, lkb_amb, denoise=True))

print('baris gazetteer di tabel utama, sebelum vs sesudah')
for nm, m in [('tier lama', m_old), ('tier baru', m_fix)]:
    print(f'  {nm:10s} P {m["precision"]:.4f}  R {m["recall"]:.4f}  F1 {m["f1"]:.4f}')
print(f'  delta F1 : {m_fix["f1"] - m_old["f1"]:+.4f}   (paper sekarang: 0,5922)')

hit = sum(1 for toks, labs in zip(gold_sents, gold_labels)
          for s, e, _ in extract_entities(labs)
          if ' '.join(toks[s:e]).lower() in shadowed)
print(f'\nentitas gold yang surface-nya keluar dari tier Unambiguous: {hit} / 1182 '
      f'({100*hit/1182:.1f}%)')

baris gazetteer di tabel utama, sebelum vs sesudah
  tier lama  P 0.4691  R 0.8029  F1 0.5922
  tier baru  P 0.7993  R 0.7716  F1 0.7852
  delta F1 : +0.1930   (paper sekarang: 0,5922)

entitas gold yang surface-nya keluar dari tier Unambiguous: 47 / 1182 (4.0%)


In [ ]:
print('btc' in lkb_unamb, 'bitcoin' in lkb_unamb, [s for s in lkb_unamb if 'harrypotter' in s or s=='betece'])

True True ['harrypotterobamasonic10inu (eth)', 'harrypotterobamasonic10inu', 'betece']


In [ ]:
m_nai_old = entity_f1(gold_labels, gazetteer_predict(gold_sents, lkb_unamb,       lkb_amb, denoise=False))
m_nai_fix = entity_f1(gold_labels, gazetteer_predict(gold_sents, lkb_unamb_fixed, lkb_amb, denoise=False))
for nm, m in [('naif tier lama', m_nai_old), ('naif tier baru', m_nai_fix)]:
    print(f'  {nm:16s} P {m["precision"]:.4f}  R {m["recall"]:.4f}  F1 {m["f1"]:.4f}')

  naif tier lama   P 0.3324  R 0.8080  F1 0.4710
  naif tier baru   P 0.3324  R 0.8080  F1 0.4710


#### A2 — regenerasi silver denoised v2

In [ ]:
# ==========================================================================
# A2a — bekukan tier fix, regenerasi silver denoised v2
# Menulis DUA file baru. silver_standard_bio.csv (ber-DOI) tidak disentuh.
# ==========================================================================
import json, re, time
import pandas as pd
from collections import Counter
from utils import gazetteer_label_tokens, extract_entities, load_bio_csv, TOKEN_PATTERN

P_TIER_REMOVAL  = f'{DATA}/lkb_tier_removal.json'
P_SILVER_BIO_V2 = f'{DATA}/silver_standard_bio_v2.csv'
OVERWRITE = False

assert OVERWRITE or not os.path.exists(P_SILVER_BIO_V2), \
    'v2 sudah ada — set OVERWRITE=True kalau memang mau menimpa'

with open(P_TIER_REMOVAL, 'w', encoding='utf-8') as f:
    json.dump(sorted(removal), f, ensure_ascii=False, indent=1)
print(f'himpunan pembuangan ({len(removal)}) -> {P_TIER_REMOVAL}')

orig_ids, _, _ = load_bio_csv(P_SILVER_BIO)
if 'df_silver_raw' not in globals():
    df_silver_raw = pd.read_csv(P_SILVER_RAW).dropna(subset=['clean_text'])
assert len(orig_ids) == len(df_silver_raw), \
    f'{len(orig_ids)} kalimat v1 vs {len(df_silver_raw)} baris raw — mapping ID tidak 1:1'

rows, n_ent = [], 0
kept, sup = Counter(), Counter()
t0 = time.time()
for sid, (_, row) in zip(orig_ids, df_silver_raw.iterrows()):
    toks = re.findall(TOKEN_PATTERN, str(row['clean_text']))
    lab = gazetteer_label_tokens(toks, lkb_unamb_fixed, lkb_amb, denoise=True)
    nai = gazetteer_label_tokens(toks, lkb_unamb_fixed, lkb_amb, denoise=False)
    n_ent += len(extract_entities(lab))
    for t, ld, ln in zip(toks, lab, nai):
        tl = t.lower()
        if tl in lkb_amb and ln != 'O':
            (sup if ld == 'O' else kept)[tl] += 1
        rows.append({'Sentence_ID': sid, 'Token': t, 'Label': ld})

pd.DataFrame(rows).to_csv(P_SILVER_BIO_V2, index=False)
print(f'{len(df_silver_raw)} kalimat, {n_ent} entitas, {time.time()-t0:.0f}s')
print(f'-> {P_SILVER_BIO_V2}')

himpunan pembuangan (179) -> /content/drive/MyDrive/crypto-ner/data/lkb_tier_removal.json
25726 kalimat, 22892 entitas, 10s
-> /content/drive/MyDrive/crypto-ner/data/silver_standard_bio_v2.csv


In [ ]:
# ==========================================================================
# A2b — jangan latih apa pun sebelum keempat assert ini lolos
# ==========================================================================
v1_ids, v1_sents, v1_labels = load_bio_csv(P_SILVER_BIO)
v2_ids, v2_sents, v2_labels = load_bio_csv(P_SILVER_BIO_V2)

n_v1 = sum(len(extract_entities(l)) for l in v1_labels)
n_v2 = sum(len(extract_entities(l)) for l in v2_labels)
print(f'kalimat  v1 {len(v1_sents)}   v2 {len(v2_sents)}')
print(f'entitas  v1 {n_v1}   v2 {n_v2}   selisih {n_v1 - n_v2}')
print(f'entitas/kalimat  v1 {n_v1/len(v1_sents):.3f}   v2 {n_v2/len(v2_sents):.3f}   gold 0.809')

assert len(v1_sents) == len(v2_sents),                'jumlah kalimat berubah'
assert n_v2 == n_den_fix,                             f'CSV {n_v2} != hitungan A1b {n_den_fix}'
assert all(a == b for a, b in zip(v1_sents, v2_sents)), 'tokenisasi berubah'
assert v1_ids == v2_ids,                              'urutan sentence ID berubah'
print('OK — token dan ID identik, hanya label yang berbeda')

diff = sum(1 for a, b in zip(v1_labels, v2_labels) if a != b)
print(f'{diff}/{len(v1_labels)} kalimat berubah labelnya '
      f'({100*diff/len(v1_labels):.1f}%)')

df_sup2 = pd.DataFrame(
    [(s, kept[s], sup[s], 100*kept[s]/max(1, kept[s]+sup[s])) for s, _ in sup.most_common(25)],
    columns=['Surface form', 'Dipertahankan', 'Ditekan', '% dipertahankan'])
df_sup2.to_csv(f'{OUT}/table_denoising_suppressed_v2.csv', index=False)
display(df_sup2.round(1))

top_kept = pd.DataFrame(
    [(s, k, sup[s], 100*k/max(1, k+sup[s])) for s, k in kept.most_common(15)],
    columns=['Surface form', 'Dipertahankan', 'Ditekan', '% dipertahankan'])
display(top_kept.round(1))
print(f'total dipertahankan {sum(kept.values())} | total ditekan {sum(sup.values())}')

kalimat  v1 25726   v2 25726
entitas  v1 37765   v2 22892   selisih 14873
entitas/kalimat  v1 1.468   v2 0.890   gold 0.809
OK — token dan ID identik, hanya label yang berbeda
10352/25726 kalimat berubah labelnya (40.2%)


,Surface form,Dipertahankan,Ditekan,% dipertahankan
0,ini,39,9115,0.4
1,ada,18,5309,0.3
2,hype,20,2055,1.0
3,amp,0,2033,0.0
4,dia,1,1375,0.1
5,sol,62,801,7.2
6,mana,5,725,0.7
7,bnb,50,710,6.6
8,gua,7,694,1.0
9,in,0,478,0.0


,Surface form,Dipertahankan,Ditekan,% dipertahankan
0,solana,509,0,100.0
1,hyperliquid,105,0,100.0
2,ath,62,391,13.7
3,sol,62,801,7.2
4,bnb,50,710,6.6
5,ini,39,9115,0.4
6,chainlink,34,0,100.0
7,uniswap,29,0,100.0
8,cardano,28,0,100.0
9,pump,24,260,8.5


total dipertahankan 1254 | total ditekan 30277


## E. Inter-annotator agreement — drawing the overlap sample

Draws **150 tweets stratified over the three strata** and writes a blank BIO
template per annotator.

Rules that keep this defensible:

* the sample is stratified, not the first 150 rows;
* templates carry **no prior labels** and are shuffled with a different seed per
  annotator, so nobody re-annotates in the original order;
* the resulting labels are used for reliability reporting **only** — the gold
  corpus is not re-adjudicated, because that would invalidate every number
  already computed against it.

In [ ]:
rng = np.random.RandomState(2026)

strata_series = pd.Series(strata)
target = {'Blue-Chip': 0.30, 'Altcoin': 0.50, 'Background': 0.20}

sample_idx = []
for s, frac in target.items():
    pool = np.where(strata_series == s)[0]
    k = min(len(pool), int(round(IAA_N * frac)))
    sample_idx.extend(rng.choice(pool, size=k, replace=False).tolist())
sample_idx = sorted(sample_idx)
print(f'IAA sample: {len(sample_idx)} tweets')
print(pd.Series([strata[i] for i in sample_idx]).value_counts().to_string())

pd.DataFrame({'index': sample_idx,
              'sentence_id': [gold_ids[i] for i in sample_idx]}).to_csv(
    P_IAA_INDEX, index=False)

for a in range(1, 4):
    order = rng.permutation(len(sample_idx))
    rows = []
    for pos in order:
        i = sample_idx[pos]
        for tok in gold_sents[i]:
            rows.append({'Sentence_ID': gold_ids[i], 'Token': tok, 'Label': ''})
        rows.append({'Sentence_ID': '', 'Token': '', 'Label': ''})
    # pd.DataFrame(rows).to_csv(f'{OUT}/iaa_template_annotator{a}.csv', index=False)
    print(f'  wrote iaa_template_annotator{a}.csv')

IAA sample: 150 tweets
Altcoin       75
Blue-Chip     45
Background    30
  wrote iaa_template_annotator1.csv
  wrote iaa_template_annotator2.csv
  wrote iaa_template_annotator3.csv


## F. Inter-annotator agreement — computation

Run after the three completed templates come back as
`iaa_annotator{1,2,3}_done.csv`.

Two numbers are reported, deliberately:

* **Fleiss' κ (token level)** is the figure reviewers expect, but it is inflated
  by the dominance of `O`.
* **Pairwise entity-level F1** is the honest measure for NER, because it only
  counts spans that both annotators actually marked.

Report both; explain the gap in one sentence.

In [ ]:
def read_done(path, keep_ids):
    df = pd.read_csv(path).dropna(subset=['Token'])
    df = df[df['Token'].astype(str).str.strip() != '']
    df['Label'] = df['Label'].fillna('O').astype(str).str.strip().replace('', 'O')
    out = {}
    for sid, g in df.groupby('Sentence_ID', sort=False):
        out[str(sid)] = g['Label'].tolist()
    return [out[s] for s in keep_ids]

sample_ids = [gold_ids[i] for i in sample_idx]
try:
    ann = [read_done(f'{OUT}/iaa_annotator{a}_done.csv', sample_ids) for a in (1, 2, 3)]

    lens = [[len(s) for s in a] for a in ann]
    assert lens[0] == lens[1] == lens[2], 'Token counts differ between annotators'

    table = token_kappa_table(ann)
    kappa = fleiss_kappa(table)
    pw = pairwise_entity_agreement(ann)

    print(f"Fleiss' kappa (token level, 3 annotators) : {kappa:.4f}")
    print('Pairwise entity-level F1 agreement:')
    for k, v in pw.items():
        print(f'  {k}: {v:.4f}')
    print(f'  mean: {np.mean(list(pw.values())):.4f}')

    pd.DataFrame([{'Metric': "Fleiss' kappa (token)", 'Value': round(kappa, 4)}] +
                 [{'Metric': f'Entity-level F1 {k}', 'Value': round(v, 4)} for k, v in pw.items()]
                 ).to_csv(f'{OUT}/table_iaa.csv', index=False)
except FileNotFoundError:
    print('Completed annotation files not found yet — run section E first, '
          'then place iaa_annotator{1,2,3}_done.csv in the analysis directory.')

Fleiss' kappa (token level, 3 annotators) : 0.9654
Pairwise entity-level F1 agreement:
  A1-A2: 0.9858
  A1-A3: 0.9489
  A2-A3: 0.9638
  mean: 0.9662


### Interpretation guide

| Fleiss' κ | Reading |
|---|---|
| > 0.80 | Almost perfect — report and move on |
| 0.61 – 0.80 | Substantial — the normal range for NER; fine |
| 0.41 – 0.60 | Moderate — say so plainly and note it as a limitation |
| < 0.40 | Investigate before submitting; usually a guideline ambiguity, most often entity boundaries around `$TICKER` and `#hashtag` forms |

Alongside the numbers, add three or four sentences to Section III.E describing
the guideline itself: how `$BTC` and `#Bitcoin` were bounded, how multi-token
project names were handled, and who adjudicated disagreements.